# Build retriever indexes (Google Colab)

This notebook **only builds** the FinQA and TAT-QA retriever indexes and downloads the bundles. **No evaluation** — no Claude API calls, no cost.

1. Clone the repo and install dependencies.
2. Download FinQA and TAT-QA datasets.
3. (Optional) Generate `first_5_rows.json` for FinQA.
4. Build **FinQA** retriever index (GPU or CPU).
4a. **Download FinQA index** — do this if you only want FinQA, then you can disconnect.
4b. Build **TAT-QA** retriever index (GPU or CPU).
4c. **Download TAT-QA index** — right after building TAT-QA.
6. **Local:** Verify indexes load (section 6).
7. **Colab only:** Disconnect and delete runtime (section 7) to release the T4 GPU and avoid extra usage.

Run every cell in order on Colab; then unzip the downloaded bundles locally for use with **demo_agentic_rag_eval.ipynb** (evaluation and display for interviews).


## Purpose

This notebook is for **building and downloading** the retriever indexes only. Use **demo_agentic_rag_eval.ipynb** to run RAG evaluation (FinQA + TAT-QA) and display results — that notebook assumes indexes are already built or downloaded.


## 1. Clone repo and install dependencies

The first code cell below also registers a **global exception handler**: if any later cell raises an error, the Colab runtime will disconnect automatically so you don't keep incurring GPU usage after a failure.


In [ ]:
REPO_URL = "https://github.com/leemingloon/ocr-agentic-rag.git"
REPO_DIR = "ocr-agentic-rag"

import os
os.environ["COLAB_GPU"] = os.environ.get("COLAB_GPU", "1")

if os.path.isdir(REPO_DIR):
    get_ipython().run_line_magic("cd", REPO_DIR)
    get_ipython().system("git pull")
else:
    get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    get_ipython().run_line_magic("cd", REPO_DIR)

!pip install -q sentence-transformers faiss-cpu rank_bm25 llama-index-core anthropic langgraph pandas

# On Colab: if any cell raises an exception, disconnect runtime to stop GPU billing.
try:
    from google.colab import runtime
    def _disconnect_on_error(etype, evalue, tb):
        try:
            runtime.unassign()
        except Exception:
            pass
        return False  # let IPython show the traceback
    get_ipython().set_custom_exc((BaseException,), _disconnect_on_error)
except Exception:
    pass  # not in Colab or API changed


## 2. Download RAG datasets (FinQA + TAT-QA)

This step uses `scripts/download_rag_datasets.py` to fetch the FinQA and TAT-QA datasets into the locations expected by the evaluation pipeline and dataset adapters.

For example, to download only FinQA you could run:

```bash
!python scripts/download_rag_datasets.py --datasets finqa
```

In the cell below we download **both** FinQA and TAT-QA.


In [ ]:
!python scripts/download_rag_datasets.py --datasets finqa tatqa


## 3. Generate first_5_rows for FinQA (optional)

This is a small convenience helper that writes out the first 5 rows of the FinQA training set to `data/rag/FinQA/train/first_5_rows.json` so you can quickly inspect the schema and example questions.


In [ ]:
import json
from pathlib import Path

train_dir = Path("data/rag/FinQA/train")
train_qa_path = train_dir / "train_qa.json"
first_5_path = train_dir / "first_5_rows.json"

if train_qa_path.exists():
    with open(train_qa_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    rows = data[:5] if isinstance(data, list) else []
    with open(first_5_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Wrote {first_5_path} ({len(rows)} rows).")
else:
    print("train_qa.json not found; run section 2 first.")


## 4. Build FinQA retriever index (GPU on Colab)

We now build a **retriever index** over the FinQA corpus using the `HybridRetriever`. **All pre-index steps run automatically before embedding** (so they are done on Colab before the index is built):

- **Section tagging:** Chunks are labeled with document section (income statement, balance sheet, notes) from context.
- **Unit parsing:** Units (millions, thousands, per-share, quarterly) are detected at chunk ingestion and stored in metadata.
- **Deduplication:** Content is hashed; duplicates are merged (first kept, `duplicate_count` set); index is built over unique chunks.
- **Provenance:** Page (from FinQA id), section, and table ID are stored per chunk for retrieval preference.

- On Colab, we default to **GPU** (`COLAB_GPU=1`). For local or CPU-only runs, use the `--cpu` flag.
- **Table-aware (Level 3):** Set `TABLE_AWARE = True` below for one chunk per table row with headers inline. After building, run **section 4a** to download and replace your local `data/rag/FinQA/train/finqa_retriever_index/`.
- **No dedup:** Set `NO_DEDUP = True` below to skip content-hash dedup so every corpus_id keeps its own table chunks (avoids missing chunks when multiple entries share the same table). Index will be larger.

The script `scripts/build_finqa_embeddings_colab.py` chunks the corpus, runs the pre-index pipeline above, then builds and saves the index bundle.

**One Colab run:** With `TABLE_AWARE = True` and `NO_DEDUP = True` (default below), a single build secures **header-per-row table-aware chunking**, **section tagging**, **unit parsing** (from chunk + doc context), **no dedup** (each corpus_id keeps its chunks), and **provenance**. See **data/proof/INDEX_CHECKLIST.md** for a verification checklist before you run.


In [ ]:
import os

# Table-aware (Level 3): one chunk per table row with column headers inline. Recommended to avoid mislabeled values (RAG_LESSONS.md). Re-download index (section 4a) and replace local bundle after building.
TABLE_AWARE = True
# Skip dedup so every corpus_id keeps its own table chunks (avoids missing chunks when multiple entries share the same table, e.g. page_64.pdf-1..4). Index will be larger.
NO_DEDUP = True

batch_size = os.environ.get("EMBED_BATCH_SIZE", "170")
use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
table_aware_flag = " --table_aware" if TABLE_AWARE else ""
no_dedup_flag = " --no_dedup" if NO_DEDUP else ""
if use_gpu:
    get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --batch_size {batch_size}{table_aware_flag}{no_dedup_flag}")
else:
    get_ipython().system(f"python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --cpu{table_aware_flag}{no_dedup_flag}")


## 4a. Download FinQA index

After building FinQA above, you can **download or save the FinQA index here** and then disconnect (e.g. run section 7). You don't have to build TAT-QA. Set **SAVE_TO_DRIVE** and **DRIVE_FOLDER** below.


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

SAVE_TO_DRIVE = False  # True = copy to your Google Drive (same as drive.google.com)
DRIVE_FOLDER = "colab_index_bundles"

def _zip_and_save(name: str, src_dir: Path, zip_name: str):
    if not src_dir.exists():
        print(f"{name}: index not found at {src_dir}; skip.")
        return
    zip_path = Path(zip_name)
    shutil.make_archive(zip_path.with_suffix(""), "zip", src_dir.parent, src_dir.name)
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount failed:", e)
            files.download(str(zip_path))
            return
        drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_name)
        print(f"{name}: saved to Drive: {drive_dir / zip_name}")
    else:
        files.download(str(zip_path))
        print(f"{name}: download triggered. Unzip into data/rag/FinQA/train/ so that finqa_retriever_index/ exists.")

_zip_and_save("FinQA", Path("data/rag/FinQA/train/finqa_retriever_index"), "finqa_retriever_index.zip")


## 4b. Build TAT-QA retriever index (GPU on Colab)

Mirrors section 4 for **TAT-QA**: the same **pre-index steps** (header-per-row chunking with `--table_aware`, section tagging, unit parsing, dedup, provenance) run in one go. Set `TABLE_AWARE = True` (same variable as in section 4) for Level 3 row-level serialization. Output: `data/rag/TAT-QA/tatqa_retriever_index/`. After building, run **section 6** to download and replace your local TAT-QA index. Verification: **data/proof/INDEX_CHECKLIST.md**.

In [ ]:
import os

# Use same TABLE_AWARE as section 4 (set there). If you only ran section 4, define it here: TABLE_AWARE = True
try:
    TABLE_AWARE
except NameError:
    TABLE_AWARE = True

batch_size = os.environ.get("EMBED_BATCH_SIZE", "170")
use_gpu = os.environ.get("COLAB_GPU", "1") == "1"
table_aware_flag = " --table_aware" if TABLE_AWARE else ""
if use_gpu:
    get_ipython().system(f"python scripts/build_tatqa_embeddings_colab.py --output data/rag/TAT-QA/tatqa_retriever_index --batch_size {batch_size}{table_aware_flag}")
else:
    get_ipython().system(f"python scripts/build_tatqa_embeddings_colab.py --output data/rag/TAT-QA/tatqa_retriever_index --cpu{table_aware_flag}")

## 4c. Download TAT-QA index

After building TAT-QA above, you can **download or save the TAT-QA index here**. Set **SAVE_TO_DRIVE** and **DRIVE_FOLDER** below (same as in section 4a).


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

SAVE_TO_DRIVE = False  # True = copy to your Google Drive (same as drive.google.com)
DRIVE_FOLDER = "colab_index_bundles"

def _zip_and_save(name: str, src_dir: Path, zip_name: str):
    if not src_dir.exists():
        print(f"{name}: index not found at {src_dir}; skip.")
        return
    zip_path = Path(zip_name)
    shutil.make_archive(zip_path.with_suffix(""), "zip", src_dir.parent, src_dir.name)
    if SAVE_TO_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print("Drive mount failed:", e)
            files.download(str(zip_path))
            return
        drive_dir = Path("/content/drive/MyDrive") / DRIVE_FOLDER
        drive_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(zip_path, drive_dir / zip_name)
        print(f"{name}: saved to Drive: {drive_dir / zip_name}")
    else:
        files.download(str(zip_path))
        print(f"{name}: download triggered. Unzip into data/rag/TAT-QA/ so that tatqa_retriever_index/ exists.")

_zip_and_save("TAT-QA", Path("data/rag/TAT-QA/tatqa_retriever_index"), "tatqa_retriever_index.zip")


**Tip — Edge stuck on download permission:** To allow Colab downloads in Microsoft Edge so they don't hang: (1) Open **edge://settings/downloads** and turn off "Ask where to save each file before downloading" if you're okay with a default folder, or (2) In **edge://settings/content** → Automatic downloads, add `https://colab.research.google.com` to allowed sites. Alternatively use **SAVE_TO_DRIVE = True** in sections 4a/4c and skip browser download entirely.


## 5. Local test: verify retriever indexes

Run this **on your local PC** after you have placed the index bundles under `data/rag/FinQA/train/` and/or `data/rag/TAT-QA/`. It loads each available index and runs a quick retrieval to confirm the bundle works (no GPU required for this test).

In [ ]:
from pathlib import Path
import sys

# Run from repo root so data/ paths resolve
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from rag_system.retrieval import HybridRetriever

def test_index(name: str, index_dir: Path, query: str, corpus_id: str = None):
    if not index_dir.exists() or not (index_dir / "meta.json").exists():
        print(f"{name}: index not found at {index_dir} (skip).")
        return
    print(f"{name}: loading from {index_dir} ...")
    retriever = HybridRetriever()
    retriever.load_index_bundle(str(index_dir))
    results = retriever.retrieve(query, top_k=2, corpus_id=corpus_id)
    print(f"  Query: {query[:60]}...")
    print(f"  Retrieved {len(results)} chunk(s).")
    if results:
        text = results[0][0].text
        print(f"  First chunk preview: {text[:150]}...")
    print()

# FinQA: expect index at data/rag/FinQA/train/finqa_retriever_index
test_index(
    "FinQA",
    repo_root / "data" / "rag" / "FinQA" / "train" / "finqa_retriever_index",
    "What was the interest expense in 2009?",
    corpus_id=None,  # or a specific doc id from your data to test corpus scoping
)

# TAT-QA: expect index at data/rag/TAT-QA/tatqa_retriever_index
test_index(
    "TAT-QA",
    repo_root / "data" / "rag" / "TAT-QA" / "tatqa_retriever_index",
    "What is the total sales amount?",
    corpus_id=None,
)

print("Done. If both indexes were found and retrieved chunks, you can run eval_runner locally.")

## 6. Disconnect and delete runtime (Colab only)

Run this cell **on Colab** after you have downloaded the index bundles (sections 4a / 4c). It disconnects and deletes the runtime so the T4 GPU is released and you stop incurring usage—helps stay within free-tier limits. If you run this notebook locally, skip this cell (it will no-op or remind you to disconnect manually in Colab).

In [ ]:
# Release the Colab runtime (disconnect + delete) to stop GPU usage.
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print("Could not unassign runtime:", e)
    print("In Colab: use Runtime → Disconnect and delete runtime to stop GPU usage.")